# Day 7: Simon's Algorithm with Qiskit

## Objective
Implement Simon's algorithm to identify a hidden three-bit binary string using a quantum circuit.

## Learning Goals
- Understand the XOR operation and Simon's promise.
- Explore superposition using Hadamard gates.
- Construct and test a quantum oracle.
- Simulate quantum measurements using Qiskit Aer.
- Recover the hidden string from measurement results.

**Example:** Hidden string `110`

**Environment:** Python, Qiskit, Google Colab.

In [15]:
%pip install -q qiskit qiskit-aer

In [16]:
import qiskit
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

print("Qiskit version:", qiskit.__version__)
print("Setup complete!")

Qiskit version: 2.6.0
Setup complete!


## 1. Classical Function and Simon's Promise

Simon's algorithm finds a hidden nonzero binary string `s`. The function satisfies `f(x) = f(y)` when `x XOR y = s` for distinct inputs.

For this example, the hidden string is `110`, and the function is:

`f(x0, x1, x2) = (x0 XOR x1, x2)`

The function takes three input bits and returns two output bits. I will verify its properties before building the quantum circuit.

In [17]:
def simon_function(x):
    if len(x) != 3 or any(bit not in "01" for bit in x):
        raise ValueError("Input must be a three-bit binary string.")

    x0, x1, x2 = map(int, x)
    return f"{x0 ^ x1}{x2}"


secret = "110"

for number in range(8):
    x = format(number, "03b")
    print(f"f({x}) = {simon_function(x)}")

for number in range(8):
    x = format(number, "03b")
    paired = format(number ^ int(secret, 2), "03b")
    assert simon_function(x) == simon_function(paired)

print("Classical Simon promise verified!")

f(000) = 00
f(001) = 01
f(010) = 10
f(011) = 11
f(100) = 10
f(101) = 11
f(110) = 00
f(111) = 01
Classical Simon promise verified!


## 2. Understanding Superposition

The Hadamard gate transforms the state `|0⟩` into an equal superposition of `|0⟩` and `|1⟩`.

Superposition allows a quantum circuit to process combinations of input states. Simon's algorithm uses Hadamard gates before and after its oracle.

In [18]:
warmup = QuantumCircuit(1)
warmup.h(0)

print(warmup.draw())
print(Statevector.from_instruction(warmup))

   ┌───┐
q: ┤ H ├
   └───┘
Statevector([0.70710678+0.j, 0.70710678+0.j],
            dims=(2,))


## 3. Constructing and Testing the Oracle

The oracle computes the function using controlled-NOT (CNOT) gates.

Qiskit displays higher-indexed qubits on the left. Therefore, input `x0 x1 x2` maps to qubits `q2 q1 q0`.

- Output `x0 XOR x1` is stored in `q4`.
- Output `x2` is stored in `q3`.

I will test the oracle using input `110` before running the full algorithm.

In [19]:
def apply_simon_oracle(circuit):
    circuit.cx(2, 4)  # x0 contributes to first output
    circuit.cx(1, 4)  # x1 contributes to first output
    circuit.cx(0, 3)  # x2 becomes second output


test = QuantumCircuit(5)

# Prepare input 110 on q2, q1, q0.
test.x(1)
test.x(2)

apply_simon_oracle(test)

state = Statevector.from_instruction(test)
index = state.probabilities().argmax()
label = format(index, "05b")

print("Oracle output state:", label)
assert label == "00110"
print("Oracle test passed!")

Oracle output state: 00110
Oracle test passed!


## 4. Running the Complete Quantum Algorithm

The algorithm follows four steps:

1. Apply Hadamard gates to the three input qubits.
2. Apply the oracle.
3. Apply Hadamard gates to the input qubits again.
4. Measure the input register.

For the hidden string `110`, each measured bit string `z` must satisfy `z · s = 0 (mod 2)`. I will simulate 1,024 shots and examine the results.

In [20]:
qc = QuantumCircuit(5, 3)

# Step 1: Create superposition.
qc.h(0)
qc.h(1)
qc.h(2)

# Step 2: Apply the oracle.
apply_simon_oracle(qc)

# Step 3: Apply Hadamard gates again.
qc.h(0)
qc.h(1)
qc.h(2)

# Step 4: Measure input qubits.
qc.measure([0, 1, 2], [0, 1, 2])

print(qc.draw())

simulator = AerSimulator()
result = simulator.run(qc, shots=1024).result()
counts = result.get_counts()

print("Measurement counts:", counts)

assert all(bits[0] == bits[1] for bits in counts), \
    "Parity check failed. Check the circuit."

print("All measurements satisfy the expected parity condition!")

     ┌───┐          ┌───┐     ┌─┐      
q_0: ┤ H ├───────■──┤ H ├─────┤M├──────
     ├───┤       │  └───┘┌───┐└╥┘   ┌─┐
q_1: ┤ H ├───────┼────■──┤ H ├─╫────┤M├
     ├───┤       │    │  ├───┤ ║ ┌─┐└╥┘
q_2: ┤ H ├──■────┼────┼──┤ H ├─╫─┤M├─╫─
     └───┘  │  ┌─┴─┐  │  └───┘ ║ └╥┘ ║ 
q_3: ───────┼──┤ X ├──┼────────╫──╫──╫─
          ┌─┴─┐└───┘┌─┴─┐      ║  ║  ║ 
q_4: ─────┤ X ├─────┤ X ├──────╫──╫──╫─
          └───┘     └───┘      ║  ║  ║ 
c: 3/══════════════════════════╩══╩══╩═
                               0  2  1 
Measurement counts: {'111': 270, '110': 247, '001': 259, '000': 248}
All measurements satisfy the expected parity condition!


## 5. Recovering the Hidden String

Each measurement gives an equation `z · s = 0 (mod 2)`.

I will test every possible nonzero three-bit string against the measured results. A candidate is valid only if it satisfies every equation. If the measurements provide enough independent information, the hidden string will be uniquely identified.

In [21]:
def dot_mod2(z, s):
    return sum(int(a) * int(b) for a, b in zip(z, s)) % 2


observed = list(counts.keys())
candidates = []

for number in range(1, 8):
    candidate = format(number, "03b")

    if all(dot_mod2(z, candidate) == 0 for z in observed):
        candidates.append(candidate)

print("Observed strings:", observed)
print("Possible hidden strings:", candidates)

if candidates == ["110"]:
    print("Hidden string uniquely recovered: 110")
elif not candidates:
    print("No candidate found. Check the circuit.")
else:
    print("More independent measurements may be needed.")

Observed strings: ['111', '110', '001', '000']
Possible hidden strings: ['110']
Hidden string uniquely recovered: 110


## 6. Results and Conclusion

### Results
I verified the classical Simon promise, tested the oracle, simulated the complete quantum circuit, and used measurement equations to recover the hidden string.

### Lessons Learned
This exercise helped me understand superposition, controlled-NOT gates, qubit ordering, quantum measurements, and binary equations in Simon's algorithm.

### Limitations
This is a small simulator-based demonstration. It illustrates the algorithm's logic but does not demonstrate a practical speed advantage on real quantum hardware. Recovery requires enough independent measurement equations.

### Conclusion
I implemented Simon's algorithm in Qiskit and recovered the hidden string `110` from simulated measurement results.